# MCP Handbook

**[Living document](https://github.com/eve-esa/mcp-tool-registry/blob/main/docs/mcp-handbook.ipynb).** This handbook is updated as EVE develops and as the MCP protocol is upgraded. The rules, examples, and figures describe the registry and the protocol as they stand today, and they will change when either of those moves.

**Who this guide is for.** Earth Observation scientists, and engineers, who want to use the Model Context Protocol (MCP) to wrap a service, automate a workflow, or make data available to AI assistants. It assumes basic Python, HTTP, and JSON. It assumes no prior knowledge of MCP.

**Contents**

- [1. What MCP is](#sec-1)
  - [1.1 Participants](#sec-1-1)
- [2. Building a server: tools, resources, and prompts](#sec-2)
  - [2.1 Create the server](#sec-2-1)
  - [2.2 Tools](#sec-2-2)
  - [2.3 Resources](#sec-2-3)
  - [2.4 Prompts](#sec-2-4)
  - [2.5 Putting the primitives together](#sec-2-5)
  - [2.6 Which primitive to use](#sec-2-6)
- [3. The EVE MCP tool registry](#sec-3)
  - [3.1 What the registry is](#sec-3-1)
  - [3.2 Where a server sits](#sec-3-2)
  - [3.3 The files, and how to use each](#sec-3-3)
  - [3.4 A working server: GDACS](#sec-3-4)
  - [3.5 A server that returns an image](#sec-3-5)
  - [3.6 A server that needs the user's credentials](#sec-3-6)
  - [3.7 A Dockerfile of your own](#sec-3-7)
  - [3.8 Testing before you open the pull request](#sec-3-8)
  - [3.9 Contribute your own server](#sec-3-9)
  - [3.10 What's next](#sec-3-10)
- [4. Advanced topics](#sec-4)
  - [4.1 How the messages travel: stdio and Streamable HTTP](#sec-4-1)
  - [4.2 Stateful and stateless servers](#sec-4-2)
  - [4.3 State across calls: return a handle](#sec-4-3)
  - [4.4 Credentials on each request](#sec-4-4)
  - [4.5 When the tool needs the user: elicitation](#sec-4-5)
  - [4.6 Tool annotations: what the call will do](#sec-4-6)
  - [4.7 Extensions to the protocol](#sec-4-7)
  - [4.8 Tool discoverability](#sec-4-8)
- [Terms](#sec-terms)
- [Resources](#sec-resources)
- [Appendix](#sec-appendix)
  - [A. Existing MCP catalogues](#sec-a)
  - [B. Possible services that could be integrated](#sec-b)


<a id="sec-1"></a>

## 1. What MCP is

**[MCP (Model Context Protocol)](https://modelcontextprotocol.io/specification/latest)** is an open protocol for connecting an AI application to external data and tools. The usual picture is **"USB-C for AI"**: one interface, instead of a custom integration for every service and every assistant.

Why use it:

- **Write an integration once.** One server works with every application that speaks MCP.
- **Add capabilities without touching the client.** Applications discover tools at run time, so a new tool is visible on the next call.
- **Give the model a clear contract.** Each tool has a name, a description, and a typed schema that the model reads before it calls.
- **Keep responsibilities apart.** The server owns access to data and actions. The application owns the model and the conversation.
- **Stay simple to build.** Messages are [JSON-RPC 2.0](https://www.jsonrpc.org/specification), but SDKs hide the JSON. You write a function. The SDK advertises it, checks the arguments, and returns the result.

![MCP connects AI applications to data sources and tools](images/mcp-simple-diagram.png)

*Fig. 1. MCP as one protocol between AI applications and outside systems. Source: [MCP introduction](https://github.com/modelcontextprotocol/modelcontextprotocol/blob/main/docs/docs/2026-07-28/getting-started/intro.mdx).*


<a id="sec-1-1"></a>

### 1.1 Participants

The MCP protocol defines the following components ([architecture overview](https://modelcontextprotocol.io/docs/learn/architecture)):

- **Host.** The AI application the person talks to. It creates one client per server.
- **Client.** A component inside the host. It keeps a single connection to one server and fetches context for the host.
- **Server.** A focused program that exposes tools, resources, and prompts.

![One MCP client per server, inside the host](images/participants.svg)

*Fig. 2. One dedicated client per server. Two clients may share a remote server. Source: the participants diagram in the [architecture overview](https://modelcontextprotocol.io/docs/learn/architecture).*


<a id="sec-2"></a>

## 2. Building a server: tools, resources, and prompts

A server offers three building blocks, called primitives. They differ in who decides to use them.

- **Tools** are functions the model can call. The **model** controls them.
- **Resources** are read-only data that gives context. The **application** controls them.
- **Prompts** are reusable instruction templates. The **user** controls them.

![An LLM reaches a tool, a resource, and a prompt through MCP](images/hf-mcp-capabilities.png)

*Fig. 3. One server, three kinds of capability. Source: [Hugging Face MCP course, Key Concepts and Terminology](https://huggingface.co/learn/mcp-course/en/unit1/key-concepts).*

The client discovers each list at run time with `tools/list`, `resources/list`, and `prompts/list`. You do not need all three. A single well-described tool that answers a real question is a complete server. Add a primitive, and every connected host can see it. You do not have to ship a new client. 

<a id="sec-2-1"></a>

### 2.1 Create the server

We are going to start by creating a new server. We will use the [Python SDK](https://github.com/modelcontextprotocol/python-sdk).

- The cells below use `FastMCP` from `mcp.server.fastmcp`. They do not call the network.
- Install the SDK first. The pin is 1.30.0. From 1.29.1 the SDK finishes its settings model at import, so the old `lifespan` warning does not appear ([python-sdk PR 3352](https://github.com/modelcontextprotocol/python-sdk/pull/3352)).
- `FastMCP("name")` creates the server. FastMCP reads the function name, the type hints, and the docstring, and turns them into what the host sees.
- A host never calls your Python functions directly. It connects a client, lists what the server offers, and calls by name.
- The SDK includes an in-memory transport for this step: `create_connected_server_and_client_session` starts the server and a real `ClientSession` in the same process, with no port.
- The helper `show` prints each result as the JSON the host receives.

```bash
pip install "mcp[cli]==1.30.0"
```


In [1]:
import json
import logging
from typing import Annotated

from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session
from pydantic import Field

# The in-memory server logs every request at INFO.
logging.getLogger("mcp").setLevel(logging.WARNING)

fire = FastMCP("Wildfire data")


def show(label: str, result) -> None:
    """Print an SDK result as the JSON the host receives."""
    body = result.model_dump(mode="json", by_alias=True, exclude_none=True)
    print(f"--- {label}")
    print(json.dumps(body, indent=2))


print("server:", fire.name)


server: Wildfire data


<a id="sec-2-2"></a>

### 2.2 Tools

Tools let a model act or fetch something: query a catalogue, call an API, compute a value. Each tool does one operation, with typed inputs. The model sees the name, the description, and the JSON schema of the arguments, and it decides whether the user's question needs that call.

**How tools work**

- `tools/list` returns the available tools with their schemas.
- `tools/call` runs one tool by name, with arguments, and returns the result.
- Arguments are validated against the schema. A failed call comes back as a result with `isError: true` and a message the model can read.

A tool definition for the wildfire server, as the host receives it:

```json
{
  "name": "get_burnt_areas",
  "description": "Return burnt-area records inside a bounding box for a date range",
  "inputSchema": {
    "type": "object",
    "properties": {
      "bbox": { "type": "string", "description": "west,south,east,north in WGS-84 degrees" },
      "start_date": { "type": "string", "description": "First day, YYYY-MM-DD" },
      "end_date": { "type": "string", "description": "Last day, YYYY-MM-DD" }
    },
    "required": ["bbox", "start_date", "end_date"]
  }
}
```

The model never sees your Python code. It sees this definition, so the description and the argument text decide whether it calls the tool correctly.

**Example: a fire question**

A user asks what burned on Evia in August 2021. The model picks the tool and fills in the arguments from the question. In Python, the same definition comes from a decorated function:

- `@fire.tool()` registers the function. The docstring is the description.
- Arguments without defaults are required.
- `Field(description=...)` puts the argument text into the schema, as in the definition above.

The data below is an illustrative sample, not a measurement. A real server would query a fire service here.


In [2]:
# Illustrative sample, not real measurements.
SAMPLE_FIRES = [
    {"fire_id": "demo-001", "date": "2021-08-04", "lon": 23.35, "lat": 38.85, "burnt_ha": 1200.0},
    {"fire_id": "demo-002", "date": "2021-08-09", "lon": 23.80, "lat": 38.40, "burnt_ha": 310.5},
    {"fire_id": "demo-003", "date": "2021-08-05", "lon": 21.00, "lat": 37.50, "burnt_ha": 95.0},
]


@fire.tool()
def get_burnt_areas(
    bbox: Annotated[str, Field(description="west,south,east,north in WGS-84 degrees")],
    start_date: Annotated[str, Field(description="First day, YYYY-MM-DD")],
    end_date: Annotated[str, Field(description="Last day, YYYY-MM-DD")],
) -> str:
    """Return burnt-area records inside a bounding box for a date range.

    Returns JSON with the number of fires and, for each one, fire_id, date and burnt_ha.
    """
    west, south, east, north = (float(value) for value in bbox.split(","))
    hits = [
        {"fire_id": f["fire_id"], "date": f["date"], "burnt_ha": f["burnt_ha"]}
        for f in SAMPLE_FIRES
        if west <= f["lon"] <= east and south <= f["lat"] <= north and start_date <= f["date"] <= end_date
    ]
    return json.dumps({"count": len(hits), "fires": hits})


async with create_connected_server_and_client_session(fire) as client:
    show("tools/list", await client.list_tools())
    show(
        "tools/call get_burnt_areas",
        await client.call_tool(
            "get_burnt_areas",
            {"bbox": "22.8285,37.9489,24.5933,39.0396", "start_date": "2021-08-01", "end_date": "2021-08-31"},
        ),
    )

--- tools/list
{
  "tools": [
    {
      "name": "get_burnt_areas",
      "description": "Return burnt-area records inside a bounding box for a date range.\n\nReturns JSON with the number of fires and, for each one, fire_id, date and burnt_ha.\n",
      "inputSchema": {
        "properties": {
          "bbox": {
            "description": "west,south,east,north in WGS-84 degrees",
            "title": "Bbox",
            "type": "string"
          },
          "start_date": {
            "description": "First day, YYYY-MM-DD",
            "title": "Start Date",
            "type": "string"
          },
          "end_date": {
            "description": "Last day, YYYY-MM-DD",
            "title": "End Date",
            "type": "string"
          }
        },
        "required": [
          "bbox",
          "start_date",
          "end_date"
        ],
        "title": "get_burnt_areasArguments",
        "type": "object"
      },
      "outputSchema": {
        "properties": {
     

Look for four things in the output:

- `tools/list` gives the host `inputSchema`, not your Python signature. That schema is all the model knows about the arguments.
- Each argument carries its description, because of `Field`.
- `tools/call` returns a `content` list and `isError`. `isError` tells the host that the call failed.
- The function returns `-> str`, and FastMCP still publishes an `outputSchema`: one `result` string. A return annotation such as a `TypedDict` or a pydantic model publishes a real schema instead.

The result is small on purpose: the model has to read it. Send a count and a few fields, not a raster.

<a id="sec-2-3"></a>

### 2.3 Resources

Resources expose data the application loads as context: file contents, a licence, a dataset description, the polygon of an area the team agreed on. The application decides what to read and how to use it. It may pass everything to the model, select a part, or search it first.

Each resource has a unique URI and a MIME type.

**How resources work**

- **Direct resources** have a fixed URI, such as `dataset://burnt-areas/licence`.
- **Resource templates** have a parameter, such as `aoi://{region}`. One template covers every agreed area.
- `resources/list` returns direct resources.
- `resources/templates/list` returns templates.
- `resources/read` returns the contents of one URI.

A resource template for the wildfire server:

```json
{
  "uriTemplate": "aoi://{region}",
  "name": "area-of-interest",
  "title": "Agreed area of interest",
  "description": "GeoJSON polygon the team agreed for a named region",
  "mimeType": "application/geo+json"
}
```

A template describes itself. A host can list it, show its title, and offer a picker for `region`.

**Example: context for a briefing**

For the wildfire assistant, the application can read these before the model starts:

- `aoi://evia`: the agreed polygon, so every tool clips to the same geometry.
- `dataset://burnt-areas/licence`: the licence and citation of the fire product.

`@fire.resource("uri")` registers one. The URI is the address. A URI with `{region}` becomes a template, and the function receives `region` as an argument. The box is the one `geocode_place("Evia island")` returned on 4 November 2026.


In [3]:
AREAS = {"evia": (22.8285, 37.9489, 24.5933, 39.0396)}


@fire.resource(
    "aoi://{region}",
    name="area-of-interest",
    title="Agreed area of interest",
    description="GeoJSON polygon the team agreed for a named region",
    mime_type="application/geo+json",
)
def area_of_interest(region: str) -> str:
    """Polygon of an agreed area, as GeoJSON."""
    key = region.strip().lower()
    if key not in AREAS:
        raise ValueError(f"Unknown area '{region}'. Known areas: {', '.join(sorted(AREAS))}.")
    west, south, east, north = AREAS[key]
    ring = [[west, south], [east, south], [east, north], [west, north], [west, south]]
    return json.dumps(
        {"type": "Feature", "properties": {"region": key}, "geometry": {"type": "Polygon", "coordinates": [ring]}}
    )


@fire.resource("dataset://burnt-areas/licence", name="burnt-areas-licence", mime_type="text/plain")
def burnt_areas_licence() -> str:
    """Licence and citation of the fire product."""
    return "Sample text. A real server states the licence and the citation of its product here."


async with create_connected_server_and_client_session(fire) as client:
    show("resources/list", await client.list_resources())
    show("resources/templates/list", await client.list_resource_templates())
    show("resources/read aoi://evia", await client.read_resource("aoi://evia"))

--- resources/list
{
  "resources": [
    {
      "name": "burnt-areas-licence",
      "uri": "dataset://burnt-areas/licence",
      "description": "Licence and citation of the fire product.",
      "mimeType": "text/plain"
    }
  ]
}
--- resources/templates/list
{
  "resourceTemplates": [
    {
      "name": "area-of-interest",
      "title": "Agreed area of interest",
      "uriTemplate": "aoi://{region}",
      "description": "GeoJSON polygon the team agreed for a named region",
      "mimeType": "application/geo+json"
    }
  ]
}
--- resources/read aoi://evia
{
  "contents": [
    {
      "uri": "aoi://evia",
      "mimeType": "application/geo+json",
      "text": "{\"type\": \"Feature\", \"properties\": {\"region\": \"evia\"}, \"geometry\": {\"type\": \"Polygon\", \"coordinates\": [[[22.8285, 37.9489], [24.5933, 37.9489], [24.5933, 39.0396], [22.8285, 39.0396], [22.8285, 37.9489]]]}}"
    }
  ]
}


The direct resource appears in `resources/list`. The template appears only in `resources/templates/list`. `resources/read` returns `contents` with the URI, the MIME type, and the text. The host, not the model, chooses when to read it.

A tool result can also carry a **resource link**: a URI the client reads later, instead of the bytes. A small image can instead come back inline. Section 3.5 covers both.

<a id="sec-2-4"></a>

### 2.4 Prompts

Prompts are reusable templates with arguments. They let the server author ship the best way to use the server: a consistent request that names the right inputs. A person asks for a prompt explicitly, usually from a menu or a slash command. The model does not fire it on its own.

**How prompts work**

- `prompts/list` returns the available prompts and their arguments.
- `prompts/get` returns the messages for one prompt, with the arguments filled in. The host places those messages in the conversation.

A prompt definition, as listed by `prompts/list`. It has a `name`, an optional `title` and `description`, and a list of `arguments`. Each argument has a `name`, an optional `description`, and `required`:

```json
{
  "name": "wildfire-briefing",
  "title": "Wildfire briefing",
  "description": "Short note on fires in a place and period, with the source of every fact",
  "arguments": [
    { "name": "place", "description": "Area the note is about", "required": true },
    { "name": "start_date", "description": "First day, YYYY-MM-DD", "required": true },
    { "name": "end_date", "description": "Last day, YYYY-MM-DD", "required": true }
  ]
}
```

What the host gets from `prompts/get` is not this definition but the filled-in messages. Each message has a `role`, `user` or `assistant`, and `content`:

```json
{
  "description": "Short note on fires in a place and period, with the source of every fact",
  "messages": [
    {
      "role": "user",
      "content": { "type": "text", "text": "Write a short wildfire briefing for Evia ..." }
    }
  ]
}
```

Message content is usually text. It can also be an image, an audio clip, a resource link, or an embedded resource. This is how a prompt can bring a resource into the conversation. This follows the [prompts specification](https://modelcontextprotocol.io/specification/2026-07-28/server/prompts).

**Example: a morning briefing**

A duty officer wants the same note every morning. They choose `wildfire-briefing` and fill in the place and the dates. The template fixes what the note must contain, and the same wording works in any host. `@fire.prompt()` registers it. A returned string becomes a user message.


In [4]:
@fire.prompt(
    name="wildfire-briefing",
    title="Wildfire briefing",
    description="Short note on fires in a place and period, with the source of every fact",
)
def wildfire_briefing(
    place: Annotated[str, Field(description="Area the note is about")],
    start_date: Annotated[str, Field(description="First day, YYYY-MM-DD")],
    end_date: Annotated[str, Field(description="Last day, YYYY-MM-DD")],
) -> str:
    """Ask for a wildfire note that cites its sources."""
    return (
        f"Write a short wildfire briefing for {place}, from {start_date} to {end_date}. "
        "State the number of fires and the burnt area in hectares. "
        "Say what recent news reports. "
        "After each fact, name the tool or resource it came from."
    )


async with create_connected_server_and_client_session(fire) as client:
    show("prompts/list", await client.list_prompts())
    show(
        "prompts/get wildfire-briefing",
        await client.get_prompt(
            "wildfire-briefing", {"place": "Evia", "start_date": "2021-08-01", "end_date": "2021-08-31"}
        ),
    )

--- prompts/list
{
  "prompts": [
    {
      "name": "wildfire-briefing",
      "title": "Wildfire briefing",
      "description": "Short note on fires in a place and period, with the source of every fact",
      "arguments": [
        {
          "name": "place",
          "description": "Area the note is about",
          "required": true
        },
        {
          "name": "start_date",
          "description": "First day, YYYY-MM-DD",
          "required": true
        },
        {
          "name": "end_date",
          "description": "Last day, YYYY-MM-DD",
          "required": true
        }
      ]
    }
  ]
}
--- prompts/get wildfire-briefing
{
  "description": "Short note on fires in a place and period, with the source of every fact",
  "messages": [
    {
      "role": "user",
      "content": {
        "type": "text",
        "text": "Write a short wildfire briefing for Evia, from 2021-08-01 to 2021-08-31. State the number of fires and the burnt area in hectares. Say w

The listing carries the argument names and whether each one is required. `prompts/get` returns `messages` with a role. The host puts them into the conversation.

<a id="sec-2-5"></a>

### 2.5 Putting the primitives together

The three primitives work best together. In the wildfire assistant, two servers are connected:

- **Fire-data server**, the one built above. The `get_burnt_areas` tool, the `aoi://{region}` and licence resources, and the `wildfire-briefing` prompt.
- **News server.** A `search_news` tool.

Each step has a different caller.

**1. The user invokes a prompt.** They pick `wildfire-briefing` and enter the arguments. The host calls `prompts/get` and adds the returned message to the conversation.

**2. The application reads resources.** It attaches `aoi://evia` and `dataset://burnt-areas/licence` from the fire-data server. The model now has the exact polygon and the product's terms, with no tool call spent on them.

**3. The model calls tools.** It reads the prompt and the context, then decides what it still needs:

- `get_burnt_areas` on the fire-data server, with the bbox of the polygon it was given.
- `search_news` on the news server, for what the press reported.

The host sends each call to the right server, because it holds one client per server (section 1.1).

**4. The model answers.** It writes the note. Each fact names the tool or resource it came from. If a tool returned nothing, the note says so.

![The user picks a prompt, the host reads a resource, and the model calls a tool](images/primitives-together.svg)

*Fig. 4. One briefing, three primitives, three different callers.*

The cell runs steps 1 to 3 against the fire-data server. No model is attached, so the tool call that the model would choose is written by hand.


In [5]:
async with create_connected_server_and_client_session(fire) as client:
    # 1. The user picks the prompt.
    prompt = await client.get_prompt(
        "wildfire-briefing", {"place": "Evia", "start_date": "2021-08-01", "end_date": "2021-08-31"}
    )
    print("1. prompt message:", prompt.messages[0].content.text[:70], "...")

    # 2. The application reads resources and hands them to the model as context.
    polygon = json.loads((await client.read_resource("aoi://evia")).contents[0].text)
    licence = (await client.read_resource("dataset://burnt-areas/licence")).contents[0].text
    print("2. resources: polygon of", polygon["properties"]["region"], "|", licence[:30], "...")

    # 3. The model calls a tool. Here the call is written by hand.
    ring = polygon["geometry"]["coordinates"][0]
    bbox = f"{ring[0][0]},{ring[0][1]},{ring[1][0]},{ring[2][1]}"
    result = await client.call_tool(
        "get_burnt_areas", {"bbox": bbox, "start_date": "2021-08-01", "end_date": "2021-08-31"}
    )
    print("3. tool result:", result.content[0].text)

1. prompt message: Write a short wildfire briefing for Evia, from 2021-08-01 to 2021-08-3 ...
2. resources: polygon of evia | Sample text. A real server sta ...
3. tool result: {"count": 2, "fires": [{"fire_id": "demo-001", "date": "2021-08-04", "burnt_ha": 1200.0}, {"fire_id": "demo-002", "date": "2021-08-09", "burnt_ha": 310.5}]}


The result is a briefing that combines context the application chose with data the model fetched, following a template the user selected. Nothing in this flow is hard-coded in the host. The tool descriptions tell the model which arguments exist, and the prompt tells it what to produce.


<a id="sec-2-6"></a>

### 2.6 Which primitive to use

- **Tool.** Use it for an action, or for a query whose arguments depend on the question. The model decides. It may have side effects, so a host may ask for approval. Example: `get_burnt_areas`.
- **Resource.** Use it for read-only context that should be available up front. The application decides. It needs no arguments from the model, and a host can load it before the first question. Example: `aoi://evia`.
- **Prompt.** Use it for a workflow people repeat, where the wording should stay stable. The user decides. Example: `wildfire-briefing`.

Rules of thumb:

- If the model has to choose it from the question, make it a tool.
- If the host should load it before the question, make it a resource.
- If a person starts it by name, make it a prompt.
- The same data can be both. The agreed polygon can be a resource, for hosts that load context, and a lookup tool, for hosts where the model asks for it.
- Start with one tool. Add the others when context or wording should travel with the server.


<a id="sec-3"></a>

## 3. The EVE MCP tool registry

In this section we present the [EVE MCP tool registry](https://github.com/eve-esa/mcp-tool-registry), how it is built, and how to contribute to it. 

<a id="sec-3-1"></a>

### 3.1 What the registry is

The [registry](https://github.com/eve-esa/mcp-tool-registry) is the catalogue of MCP servers for Earth Observation and Earth Science that [EVE](https://eve.philab.esa.int/) uses. The EVE team maintain it, and anyone can contribute: companies, research institutes, students, and domain experts who want their favourite service available to EVE.

Four principles:

- **Quality-gated.** Every contribution passes automated checks and a code review before it is merged.
- **Platform-integrated.** A merged server is run, tested on staging, and then its tools become available in EVE.
- **MCP-compliant.** Every server follows the MCP standard, so it also works in Claude, Cursor, or any other MCP host.
- **Self-contained.** Each server is one folder with its own requirements. Copy the folder into another project and it runs.

The EVE project deploys the MCP server, not the service behind it. If your server calls a model, a database, or an API, the provider of that service runs and maintains it.

![EVE holds one MCP client per registry server, and each server calls its own upstream service](images/registry-architecture.svg)

*Fig. 5. EVE is the host. It holds one MCP client per server in the registry. In each registry server, the first line is the server name, the folder under `servers/`. The lines under it are that server's tools. Each server calls its own upstream service.*

Discovery is what makes this work. A client asks a server what it offers, then calls. Adding a tool therefore needs no change in EVE: on the next `tools/list`, the model sees the new tool and can choose it. After a merge, EVE refers to a server by its folder name under `servers/`, the same way it calls `effis` or `serpapi`.

![A pull request passes checks and code review, or the review sends an error back to that pull request. After the merge the server runs, is tested on staging, and only then is public in EVE](images/registry-lifecycle.svg)

*Fig. 6. A code review can send the pull request back. After the merge the server runs, is tested on staging, and only then is public in EVE.*

How a server goes from the registry to the platform, in the same order as the figure:

- The folder is `servers/your-server/`, with `server.py`, `requirements.txt`, and `CITATION.cff`.
- You open a pull request against `main`.
- PR checks run (`pr-checks.yml`). A failure returns to the server folder. A pass goes on to code review.
- A person reviews the code. A rejection returns to that pull request. A pass merges to `main`.
- The server runs, deployed and running.
- A person tests it on staging.
- It is public in EVE. `tools/list` shows it.


<a id="sec-3-2"></a>

### 3.2 Where a server sits

The repository is a monorepo. Each server is one folder under `servers/`. Everything else is shared infrastructure that a contributor does not touch:

```
.github/workflows/
    pr-checks.yml       # validates every pull request
    deploy.yml          # deploys the changed server after a merge
    sync-readmes.yml    # writes the server README citation section after a merge
scripts/
    validate_pr.py      # the PR checks, runnable locally
    citation.py         # validates CITATION.cff and renders the README section
    deploy_server.py    # build and deploy logic, used by CI
shared/
    Dockerfile          # the image used unless a server ships its own
servers/
    dummy_image/        # an inline image: section 3.5
    effis/              # credentials and a custom image: sections 3.6 and 3.7
    gdacs/              # the working server in section 3.4
    your-server/        # your new server goes here
```

Section 3.3 is the status of each file inside `servers/your-server/`, and how to fill it in.


<a id="sec-3-3"></a>

### 3.3 The files, and how to use each

CI refuses a pull request that breaks these rules. The [Quick Start](../CONTRIBUTING.md#quick-start) skeleton already has this shape.

| File | Status |
|---|---|
| `server.py` | required |
| `requirements.txt` | required |
| `CITATION.cff` | required |
| `.env.template` | only when the server needs a secret |
| `test.py` | recommended |
| `README.md` | recommended |
| `Dockerfile` | optional |

---


#### `server.py`

*Required.*

The file the container runs. One tool is enough. Section 3.4 runs a real file.

```python
mcp = FastMCP("My Server", host="0.0.0.0", port=8000, stateless_http=True)

@mcp.tool()
async def my_tool(query: str) -> str:
    """One sentence on when to call this tool, and what the result contains."""
    return query

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--transport", choices=["stdio", "http"], default="http")
    args = parser.parse_args()
    if args.transport == "stdio":
        mcp.run(transport="stdio")
    else:
        mcp.run(transport="streamable-http")
```

---


#### `requirements.txt`

*Required.*

One package per line, and every line has a version.

```
mcp[cli]==1.30.0
httpx>=0.27.0
python-dotenv>=1.0.0
```

---


#### `CITATION.cff`

*Required.*

It credits the authors, in [Citation File Format](https://citation-file-format.github.io/) 1.2.0. [`servers/effis/CITATION.cff`](../servers/effis/CITATION.cff) is a real one. The [cffinit](https://citation-file-format.github.io/cffinit/) form writes one for you.

The authors you list here are how a pull request is credited:

- In the repository, the file itself names them. After the merge, CI writes an Authors block into the README from this file.
- On the platform, EVE keeps the same metadata with the server. The UI and the API (`GET /mcp-servers`) show the authors there, next to the tools.

```yaml
cff-version: 1.2.0
message: If you use this MCP server, please cite it using the metadata from this file.
type: software
title: My Server
abstract: >-
  One paragraph on what the server does.
authors:
  - given-names: Jane
    family-names: Doe
    alias: janedoe                  # your GitHub username
    affiliation: University of Somewhere
license: Apache-2.0
repository-code: https://github.com/eve-esa/mcp-tool-registry/tree/main/servers/my-server
```

---


#### `.env.template`

*Only when the server needs a secret.*

Placeholders, never values. Section 3.6 shows how to correctly use it.

```
MY_API_KEY=${MY_API_KEY}
MY_API_URL=https://api.example.com
```

---


#### `test.py`

*Recommended.*

The tests for your server. Connect a real MCP client to your `mcp` object, then assert on what it returns: the tool names, the result for a good input, and `isError: true` for a bad one. A script that only calls the tools and prints the output is not a test. It never fails. `servers/gdacs/test.py` is a working example. Section 3.8 shows three ways to write one.

---


#### `README.md`

*Recommended.*

A normal README for people who find your server: what it does, the tools and their arguments, the credentials it needs, and an example call. Write it as for any project. CI does not check it. After the merge, the sync workflow adds an Authors and Citation block built from `CITATION.cff`, between `<!-- BEGIN GENERATED CITATION -->` and `<!-- END GENERATED CITATION -->`. It leaves the text outside those markers alone, and creates the file from the CFF title and abstract if you did not write one. Edit the CFF file, not the text between the markers.

---


#### `Dockerfile`

*Optional.*

**Additional setup that replaces the normal one.** Use it when the server needs a build step the shared image does not do, such as downloading a large data file. When this file exists, the deploy builds it instead of `shared/Dockerfile`. It does not add to the shared one. Your file must therefore install `requirements.txt`, copy the folder, expose port 8000 and run `server.py` itself. Extra modules stay in this folder and are copied into the image. Section 3.7 shows why EFFIS has one.


<a id="sec-3-4"></a>

### 3.4 A working server: GDACS

If you want a complete server you can run in this notebook, [GDACS](https://www.gdacs.org/) is a small public one. [`servers/gdacs/`](../servers/gdacs/) wraps the Global Disaster Alert and Coordination System API in one tool. It needs two files. The cells below write both into a temporary folder, install the requirements, start the server, and call it. Nothing outside this notebook is needed.


In [7]:
import tempfile
from pathlib import Path

server_dir = Path(tempfile.mkdtemp()) / "gdacs"
server_dir.mkdir()

**`server.py`**


In [8]:
%%writefile {server_dir}/server.py
"""
GDACS (Disasters) MCP Server
============================
An MCP server providing global natural disaster and hazard monitoring
via GDACS (Global Disaster Alert and Coordination System), a joint
initiative of the UN and European Commission. Use to search for
natural disaster events worldwide covering earthquakes, floods,
volcanoes, wildfires, droughts, cyclones and tsunamis.

Tools:
    search_gdacs_events — search ongoing/recent disasters by hazard
                        type, alert level, and/or country

Usage:
    python server.py                              # stdio transport
    python server.py --transport http --port 8000  # HTTP transport

Requirements:
    pip install "mcp[cli]==1.30.0" httpx python-dotenv
"""

from __future__ import annotations

import json
import logging
import sys
from pathlib import Path
from typing import Any, Literal

import httpx
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    stream=sys.stderr,
)

logger = logging.getLogger("disasters-mcp")

GDACS_SEARCH_URL = "https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH"
TIMEOUT = 60
GDACS_USER_AGENT = "DisastersMCPServer"

_SERVER_DIR = Path(__file__).resolve().parent
load_dotenv(_SERVER_DIR / ".env", override=False)

mcp = FastMCP("Disasters", host="0.0.0.0", port=8000, stateless_http=True)


async def _http_get(
    url: str,
    params: dict[str, Any] | None = None,
    headers: dict[str, str] | None = None,
) -> httpx.Response:
    """Perform an async HTTP GET with sensible defaults."""
    async with httpx.AsyncClient(timeout=TIMEOUT, follow_redirects=True) as client:
        resp = await client.get(url, params=params, headers=headers)
        resp.raise_for_status()
        return resp


EventType = Literal["EQ", "TC", "FL", "VO", "WF", "DR", "TS"]
AlertLevel = Literal["Green", "Orange", "Red"]


@mcp.tool()
async def search_gdacs_events(
    eventlist: EventType | None = None,
    alertlevel: AlertLevel | None = None,
    country: str | None = None,
) -> str:
    """Search GDACS for ongoing or recent natural disaster events worldwide.

    Use this when the user asks about current or recent disasters, hazards,
    or emergencies — e.g. "any active volcanoes right now", "is there
    flooding in Bangladesh", "recent earthquakes near Japan". The data comes
    from the Global Disaster Alert and Coordination System (GDACS) and is
    not limited to any one country.

    Args:
    eventlist: The type of hazard to search for. One of:
        - "EQ": seismic events
        - "FL": river/flash flooding
        - "VO": volcanic eruptions or unrest
        - "WF": forest/bush fires
        - "DR": prolonged water shortage events
        - "TC": hurricanes, typhoons, cyclones
        - "TS": tsunami warnings/events
        Leave unset to return any/all categories of hazard.
    alertlevel: Optional severity filter. "Green" = low impact,
        "Orange" = medium impact, "Red" = high impact/major disaster.
        Leave unset to return events of any severity.
    country: Optional country name to restrict results to
        (e.g. "Philippines"). Leave unset to search globally.

    Returns:
        JSON string with matching events, including event name, country,
        alert level, coordinates, and event date range.
    """

    try:
        resp = await _http_get(
            GDACS_SEARCH_URL,
            params={
                "eventlist": eventlist,
                "alertlevel": alertlevel,
                "country": country,
            },
            headers={"User-Agent": GDACS_USER_AGENT},
        )
        if not resp.text:
            # GDACS: 204 No Content with an empty body for no results
            return json.dumps({"error": "No results found for query."})
        results = resp.json()
    except Exception as exc:
        return json.dumps({"error": f"Disaster search failed: {exc}"})

    if not results.get("features"):
        return json.dumps({"error": "No results found for query."})

    return json.dumps(results)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Disasters MCP Server")
    parser.add_argument(
        "--transport",
        choices=["stdio", "http"],
        default="http",
    )
    parser.add_argument("--port", type=int, default=8000)
    args = parser.parse_args()

    mcp.settings.port = args.port

    if args.transport == "stdio":
        mcp.run(transport="stdio")
    else:
        mcp.run(transport="streamable-http")


**`requirements.txt`**


In [9]:
%%writefile {server_dir}/requirements.txt
mcp[cli]==1.30.0
httpx>=0.27.0
python-dotenv>=1.0.0


In [10]:
%pip install --quiet -r {server_dir}/requirements.txt

Start the server over HTTP, as a deployment does. Connect a client, list the tools, and ask for current floods.


In [11]:
import json
import socket
import subprocess
import sys
import time

import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# Ask the OS for a free port, so the cell does not collide with another process.
with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
url = f"http://127.0.0.1:{port}/mcp"

server = subprocess.Popen(
    [sys.executable, "server.py", "--transport", "http", "--port", str(port)],
    cwd=server_dir,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
try:
    for _ in range(50):
        try:
            httpx.get(url)
            break
        except httpx.ConnectError:
            time.sleep(0.2)

    async with streamable_http_client(url) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            result = await session.call_tool("search_gdacs_events", {"eventlist": "FL"})
finally:
    server.terminate()
    server.wait()

print("tools:", [tool.name for tool in tools.tools])
events = json.loads(result.content[0].text).get("features", [])
print("floods:", len(events))
for event in events[:3]:
    print("-", event["properties"]["name"], event["properties"]["alertlevel"])

[11/06/26 14:24:16] INFO     HTTP Request: GET http://127.0.0.1:50234/mcp "HTTP/1.1 406 Not         ]8;id=943437;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=943438;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             Acceptable"                                                                           

                    INFO     HTTP Request: POST http://127.0.0.1:50234/mcp "HTTP/1.1 200 OK"        ]8;id=943446;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=943447;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:50234/mcp "HTTP/1.1 202 Accepted"  ]8;id=943454;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=943455;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:50234/mcp "HTTP/1.1 200 OK"        ]8;id=943462;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=943463;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:50234/mcp "HTTP/1.1 200 OK"        ]8;id=943470;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=943471;file:///private/tmp/hbvenv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

tools: ['search_gdacs_events']
floods: 17
- Flood in India Orange
- Flood in China Orange
- Flood in Nepal Red


<a id="sec-3-5"></a>

### 3.5 A server that returns an image

If your tool draws or fetches a picture, you can hand it back so EVE can show it in the chat. A path inside the container is not a result the client can open.

[`servers/dummy_image`](../servers/dummy_image/server.py) is the small case. `get_sample_image` draws a solid PNG with Pillow and returns it inline as [`ImageContent`](https://modelcontextprotocol.io/specification/2026-07-28/server/tools): base64 bytes and `image/png`. EVE stores the image and shows it in the chat.

```python
@mcp.tool()
def get_sample_image(color: str = "blue", size: int = 512) -> ImageContent:
    """Generate a solid-color square PNG and return it inline as base64."""
    side = max(1, min(int(size), 2048))
    img = Image.new("RGB", (side, side), color=color)
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    return ImageContent(
        type="image",
        data=base64.b64encode(buffered.getvalue()).decode("utf-8"),
        mimeType="image/png",
    )
```

EVE then takes the picture out of the tool result and shows it. The steps are in [`ArtifactInterceptor`](https://github.com/eve-esa/backend/blob/main/src/services/mcp/artifact_ingestion.py):

- A small picture arrives as `ImageContent`. EVE decodes the base64.
- A larger file arrives as a resource link. EVE reads the bytes with `resources/read`.
- Either way, the bytes are stored as an artifact. The model sees a short stub: a markdown image and the artifact id.
- The chat renders the image from that artifact. The model does not have to read the pixels.

![A tool image is stored as an artifact, shown in the chat, and replaced by a stub for the model](images/image-transmission.svg)

*Fig. 7. A picture leaves the tool as inline bytes or as a resource link. EVE stores the bytes and shows the image. The model receives a short stub. Source: [file-return convention](https://github.com/eve-esa/backend/blob/main/docs/mcp-file-return-convention.md).*

Use an inline image when the picture is small, on the order of a few tens of kilobytes. A chart, a satellite strip, a CSV, or a PDF is a [resource link](https://modelcontextprotocol.io/specification/2026-07-28/server/tools) instead. Keep a container path out of the text.


<a id="sec-3-6"></a>

### 3.6 A server that needs the user's credentials

If your server needs credentials, because the service behind it is gated, you can let each person supply their own. EVE asks for the values and sends them on every call.

Keep a secret out of the tool arguments. Arguments reach the model, the logs, and the conversation history.

EFFIS is one server that does this. [`servers/effis/server.py`](../servers/effis/server.py) has three tools. `geocode_place` and `get_effis_burnt_areas` need no account. `compute_metrics` downloads Sentinel-2 imagery, and that needs a Copernicus Data Space account. That account can be the user's, rather than one shared key on the server.

Declare the names in the server folder. EVE then asks each user for their own values, and passes them to the server on every call. Three steps:

![How a user's credentials reach the server](images/credentials.svg)

*Fig. 8. A server with `.env.template` is marked as needing credentials. The example variable is `CDSE_CLIENT_ID`. The user adds it on the platform. If a tool is called without it, EVE asks on the spot.*

**1. Declare the credentials in `.env.template`.** One line per secret, with a `${PLACEHOLDER}` value. EFFIS declares two:

```
CDSE_CLIENT_ID=${CDSE_CLIENT_ID}
CDSE_CLIENT_SECRET=${CDSE_CLIENT_SECRET}
```

This file is what makes the rest work. When the server is deployed, EVE reads `.env.template` and knows that the server requires these secrets. The platform then lets each user add their own values, and nobody writes a key into the code. Use clear names, and say in the tool description which account the user needs.

EVE sends the user's values as HTTP headers, so list those header names in `DEFAULT_HEADER_ALLOWLIST`, in [`scripts/deploy_server.py`](../scripts/deploy_server.py). A header that is not listed is dropped before it reaches the server. EFFIS uses `X-CDSE-Client-Id` and `X-CDSE-Client-Secret`.

**2. Read them in the server.** Take the header from the current request. Fall back to the environment, which is what a local run uses. Header names arrive in lower case:

```python
def _get_request_headers() -> dict[str, str]:
    """Return HTTP headers from the current MCP request, or {} on stdio."""
    try:
        ctx = mcp.get_context()
        request = ctx.request_context.request
        if request is not None and hasattr(request, "headers"):
            return dict(request.headers)
    except Exception:
        pass
    return {}


def _resolve_cdse_creds() -> tuple[str, str]:
    """Prefer the user's headers, then the server environment."""
    headers = _get_request_headers()
    cid = headers.get("x-cdse-client-id", "") or os.getenv("CDSE_CLIENT_ID", "")
    csec = headers.get("x-cdse-client-secret", "") or os.getenv("CDSE_CLIENT_SECRET", "")
    return cid, csec
```

Resolve them once, at the start of the tool, and pass the values on. The request context may be gone by the time a later callback runs.

**3. Use them, and say what is missing.** When nothing is set, `compute_metrics` stops with a message the model can pass on to the user. Otherwise it exchanges the id and the secret for an access token. The token is what the imagery calls send. The id and the secret never leave the server:

```python
cdse_cid, cdse_csec = _resolve_cdse_creds()
if not cdse_cid or not cdse_csec:
    return _tool_error(
        "cdse_auth",
        "CDSE credentials not set. Either send them as HTTP "
        "headers (X-CDSE-Client-Id / X-CDSE-Client-Secret) or "
        "set CDSE_CLIENT_ID / CDSE_CLIENT_SECRET env vars on the server.",
    )

resp = await client.post(
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    data={"grant_type": "client_credentials", "client_id": cdse_cid, "client_secret": cdse_csec},
)
resp.raise_for_status()
token = resp.json()["access_token"]
```

**Testing it without EVE.** Send the headers yourself. A Python client attaches them to the server connection, not to a tool call. [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters) takes them in the server config:

```python
"effis": {
    "transport": "streamable_http",
    "url": "http://localhost:8000/mcp",
    "headers": {
        "X-CDSE-Client-Id": my_client_id,
        "X-CDSE-Client-Secret": my_client_secret,
    },
}
```

The Copernicus credentials come from [dataspace.copernicus.eu](https://dataspace.copernicus.eu/): register, then create an OAuth client in the [Sentinel Hub dashboard](https://shapps.dataspace.copernicus.eu/dashboard/#/account/settings).


<a id="sec-3-7"></a>

### 3.7 A Dockerfile of your own

If your server needs a setup step the shared image does not do, you can ship your own Dockerfile. Most servers can stay with the shared one.

The shared image is [`shared/Dockerfile`](../shared/Dockerfile). It installs `requirements.txt`, copies the folder, and runs `server.py` on port 8000:

```dockerfile
FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["python", "server.py"]
```

```bash
docker build -f shared/Dockerfile servers/gdacs/
```

The deploy uses a different file when the server folder contains one. [`scripts/deploy_server.py`](../scripts/deploy_server.py) picks `servers/<name>/Dockerfile` if that file exists, and `shared/Dockerfile` otherwise.

EFFIS needs that. Historical queries read a burnt-area shapefile, about 200 MB, and downloading it on every call would be too slow. [`servers/effis/Dockerfile`](../servers/effis/Dockerfile) keeps the shared layout and adds two build steps. It installs `curl` and `unzip`, then fetches the shapefile into `effis_layer/` and checks the archive arrived:

```dockerfile
RUN apt-get update \
    && apt-get install -y --no-install-recommends curl ca-certificates unzip \
    && rm -rf /var/lib/apt/lists/*

ARG EFFIS_SHAPEZIP_URL="https://maps.effis.emergency.copernicus.eu/effis?service=WFS&request=GetFeature&typename=ms:modis.ba.poly&version=1.1.0&outputformat=SHAPEZIP"
RUN mkdir -p effis_layer \
    && curl -fSL --connect-timeout 60 --max-time 1800 \
        "${EFFIS_SHAPEZIP_URL}" -o /tmp/effis_modis.poly.zip \
    && unzip -oq /tmp/effis_modis.poly.zip -d effis_layer \
    && test -f effis_layer/modis.ba.poly.shp
```

The real file also retries the same download over HTTP when HTTPS fails. The zip is not committed. The 5 MB limit applies to files in the pull request, not to what the build downloads.

```bash
docker build -f servers/effis/Dockerfile servers/effis/
```

The first build is slow: the shapefile download is allowed up to 30 minutes. Later builds reuse the layer as long as the Dockerfile and the folder do not change.


<a id="sec-3-8"></a>

### 3.8 Testing before you open the pull request

Before you open the pull request, you can run the same checks a host will make. Three ways, from fastest to closest to production. `servers/gdacs/test.py` is a full example of the first.

**In `test.py`, over HTTP.** This is what the test files in this registry do. The script starts `server.py` as a subprocess, connects a real MCP client over Streamable HTTP, and asserts on the answers. A script that only prints the result is not a test: it passes even when the tool is broken.

```python
"""Integration test. Starts the server on HTTP and calls the tools through the protocol.

    python test.py
"""
import asyncio
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

SERVER_PY = Path(__file__).resolve().parent / "server.py"
PORT = 9300

def start_server(port: int) -> subprocess.Popen:
    return subprocess.Popen(
        [sys.executable, str(SERVER_PY), "--transport", "http", "--port", str(port)],
        env=os.environ.copy(),
    )

async def wait_for_server(url: str, timeout: float = 15) -> None:
    """Poll until the server accepts connections."""
    deadline = time.monotonic() + timeout
    async with httpx.AsyncClient() as client:
        while time.monotonic() < deadline:
            try:
                if (await client.get(url)).status_code < 500:
                    return
            except httpx.ConnectError:
                pass
            await asyncio.sleep(0.3)
    raise TimeoutError(f"Server did not start within {timeout}s")

async def run_tests(port: int) -> None:
    async with httpx.AsyncClient(timeout=httpx.Timeout(60)) as http:
        async with streamable_http_client(
            f"http://localhost:{port}/mcp", http_client=http
        ) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()

                names = {t.name for t in (await session.list_tools()).tools}
                assert "my_tool" in names

                good = await session.call_tool("my_tool", {"query": "Evia"})
                assert not good.isError
                assert "Evia" in good.content[0].text

                # A bad call must fail with a message the model can act on.
                bad = await session.call_tool("my_tool", {"query": ""})
                assert bad.isError

def main() -> None:
    proc = start_server(PORT)
    try:
        asyncio.run(wait_for_server(f"http://localhost:{PORT}/mcp"))
        asyncio.run(run_tests(PORT))
        print("all tests passed")
    finally:
        if proc.poll() is None:
            proc.send_signal(signal.SIGTERM)
            proc.wait(timeout=5)

if __name__ == "__main__":
    main()
```

Assert on what the host sees: the tool names, `isError`, and the shape of the result. Add one call that should fail, and check that its text tells the model what to do next. The server must accept `--port`, as the servers in this registry do.

**With the MCP Inspector.** The [Inspector](https://modelcontextprotocol.io/docs/tools/inspector) is the reference client, with a browser UI that shows every JSON-RPC message. It needs Node 22.19 or newer. Start your server on HTTP, then connect:

```bash
python server.py                     # Streamable HTTP on port 8000
npx @modelcontextprotocol/inspector --server-url http://localhost:8000/mcp --transport http
```

Open the Tools tab, run each tool once with the arguments a model would send, and read the description as the model reads it.

**Over HTTP with curl.** [CONTRIBUTING.md](../CONTRIBUTING.md#local-testing) has the `tools/list` and `tools/call` requests. Then build the image with `shared/Dockerfile`, as CI does. A server that ships its own Dockerfile is built from that file instead (section 3.7).


<a id="sec-3-9"></a>

### 3.9 Contribute your own server

When the server is ready, you can follow the same path GDACS took. [CONTRIBUTING.md](../CONTRIBUTING.md) is the full guide. Section 3.3 is the file-by-file version.

```bash
git clone https://github.com/eve-esa/mcp-tool-registry.git
cd mcp-tool-registry
git switch -c feat/my-server
mkdir servers/my-server
```

Write the files from section 3.3. Copy `servers/gdacs/server.py` when the server is one tool over a public API, and keep the `FastMCP(...)` line and the `__main__` block. Add `.env.template` when the server needs a secret, and follow section 3.6. Add a Dockerfile only when the shared image is not enough (section 3.7).

Test it (section 3.8), then build the image the deploy will build:

```bash
cd servers/my-server && python test.py && cd ../..
docker build -f shared/Dockerfile servers/my-server/
```

Open the pull request against `main`, one server per pull request, with the checklist in the description. CI checks:

- One server folder.
- `server.py`, `requirements.txt`, and a valid `CITATION.cff`.
- The `server.py` rules in section 3.3.
- A version constraint on every requirement.
- No `.env`, key, or credential file, and no hard-coded secret.
- `${PLACEHOLDER}` syntax in `.env.template`.
- No file over 5 MB.
- `ruff`, and a Docker build.

`python scripts/validate_pr.py --files <changed files>` runs the same checks locally, after `pip install -r scripts/requirements.txt`.

A few habits make the hand-off to EVE work:

- The description says when to use the tool, what it returns, and what a failure looks like. The model writes the next call from that text.
- A failure sets `isError: true`, with a message the model can act on.
- The result stays small: a bounding box, a table of hectares, a scene id. A small picture can be an inline image. A larger file is a resource link (section 3.5), not a container path.
- Put your name in `CITATION.cff`. A merged pull request credits those authors in the repository, and in EVE, where the UI and the API show them with the server metadata (section 3.3).


<a id="sec-3-10"></a>

### 3.10 What's next

The registry is the catalogue EVE uses today. The platform can grow from here:

- More servers, so more Earth Observation services sit beside the ones already in `servers/`.
- Skills: reusable instructions a person can turn on for a task, next to the servers they already chose.
- Subagents: smaller agents that take one job and call only the tools that job needs.
- Prompts shipped with a server, offered in the chat as ready-made requests, in the same way `wildfire-briefing` works in section 2.4.


<a id="sec-4"></a>

## 4. Advanced topics

Nothing here is needed for a first server. Sections 4.1 to 4.5 explain how messages travel between host and server, and how state, credentials, and questions to the user fit in. Sections 4.6 to 4.8 cover features to add once the basics work, including how a host finds the right tool.


<a id="sec-4-1"></a>

### 4.1 How the messages travel: stdio and Streamable HTTP

So far the host and the server have exchanged messages without asking how those messages get from one to the other. That channel is the **transport**. MCP defines two standard transports. The messages are identical on both, so your tools do not change. The choice decides where your server runs and who can reach it.

**Stdio: a local server**

- The host starts your program as a subprocess on the user's computer. It sends requests on the program's standard input and reads responses from its standard output.
- Logs go to standard error. Anything else printed on standard output corrupts the message stream.
- The server is private to one user and one host. It can read that user's files and use software installed on that computer. It needs no network, no port, and no deployment.
- **Files the tool creates are written on that same computer.** The host and the user can open them directly.

Use stdio when:

- you are developing or testing a server,
- the tool needs local files, local software, or the user's own environment,
- the data must stay on the user's computer,
- one person uses the server.

**Streamable HTTP: a remote server**

- The server is a web service that runs on **another machine**: a cloud server, a container, a cluster. It keeps running whether or not the user's computer is on.
- The client sends each request as an HTTPS POST to one endpoint of that server. Use HTTPS for every remote server: requests carry tokens and data across the network. The server can answer with a stream of messages (server-sent events) when it has progress or several results to return.
- Many users and applications can share one deployment, and you update it in one place.
- **Files the tool creates stay on the server's disk.** The host can still read them, but not by opening a path on that machine. The tool returns a link, and the host asks the server for the file's content (`resources/read`, section 3.5).

Use Streamable HTTP when:

- a team or several applications should share one server,
- the work is heavy: large downloads, long computations, datasets that live near the server,
- secrets and credentials must stay on the server and not on every laptop,
- you want to update, monitor, and secure the server in one place.

The Python SDK serves plain HTTP. For a remote server, put it behind something that terminates TLS, such as a reverse proxy or a load balancer, and publish an `https://` URL. Plain `http://127.0.0.1` is fine on your own machine.

The price of a remote server is that you host it, secure it with authentication (section 4.4), and accept that it cannot see files on the user's computer.

![Stdio keeps host, server, and files on one machine. Streamable HTTP keeps the server and its files on another machine, reached over the network](images/transports.svg)

*Fig. 9. Stdio keeps host, server, and the files the tool creates on one machine. Streamable HTTP puts the server and its files on another machine. The host still reads those files, through the server.*

The same server code runs on either transport. Only the last line changes:

```python
mcp = FastMCP("wildfire-data")

mcp.run(transport="stdio")              # local: the host starts this process
# mcp.run(transport="streamable-http")  # remote: serves http://127.0.0.1:8000/mcp locally; add TLS in front
```

A common path is to develop and test over stdio, then deploy the same server over Streamable HTTP when others need it.


<a id="sec-4-2"></a>

### 4.2 Stateful and stateless servers

A second choice applies to remote servers: does the server remember earlier requests from the same client?

**Stateful**

- The server opens a session when a client connects and keeps it in the memory of that process. Later requests must reach the same process.
- This is the default for Streamable HTTP in the Python SDK. A stdio server behaves this way too: one process, one client.
- Good for: keeping per-client data in memory, and sending messages to the client outside a request.
- Costs: a follow-up request must reach the same instance. Running several instances behind a load balancer becomes hard, and a restart loses every session.

**Stateless**

- Each request stands alone and carries everything the server needs. Any instance can answer any request.
- Good for: scaling by adding instances behind a load balancer, and restarting without losing anything.
- Costs: the server holds no memory of the client. Anything that must survive between calls goes into a tool argument or into a store shared by all instances (section 4.3).

In the Python SDK, stateless mode is one argument:

```python
mcp = FastMCP("wildfire-data", stateless_http=True)
mcp.run(transport="streamable-http")
```

Which to use:

- **Stateless** is the default choice for tools that answer from their arguments, which is most of them. It is required in practice when more than one instance runs.
- **Stateful** fits a single-instance server that keeps per-client data in memory or sends messages to the client on its own.
- Write tools as if the server were stateless. They then work on every transport and every deployment. The agent keeps the conversation. Your tool answers from its arguments, or from a handle it minted earlier (section 4.3).


<a id="sec-4-3"></a>

### 4.3 State across calls: return a handle

<div class="alert alert-block alert-info" style="background-color: #e8f1f0; color: #173245; border-left: 5px solid #346562; border-radius: 6px; padding: 10px 16px; margin: 12px 0;">

**Use case.** A burn-severity analysis downloads satellite composites from before and after a fire. That can outlast one HTTP request.

</div>

Split it: one tool starts the job and returns an id, a second tool reads the job by id. This is the [stateful tools](https://modelcontextprotocol.io/specification/2026-07-28/server/tools) pattern in the specification. On the wire a handle is an ordinary string.

![The first call saves a job in a shared store and returns its id; the second call reaches another instance, which loads the job by that id](images/state-handles.svg)

*Fig. 10. A handle survives the change of instance because the state lives in a shared store, keyed by the handle.*

The specification's advice for a handle: opaque and hard to guess, a lifetime stated in the tool description, an authorisation check on every call, and a clear error when it expired. The cell follows it. It opens a separate connection for each call, the way two instances would see them.


In [12]:
import secrets
import time
from typing import Annotated, TypedDict

from mcp.server.fastmcp.exceptions import ToolError
from mcp.types import ToolAnnotations

jobs = FastMCP("Burn analysis jobs")

# Stands in for S3, DynamoDB, or Redis. A dict lives in one process and is not
# shared between instances, so a deployed server keeps jobs outside the process.
JOB_STORE: dict[str, dict] = {}
JOB_TTL_S = 24 * 3600

class JobTicket(TypedDict):
    analysis_id: str
    status: str
    expires_in_hours: int

class JobStatus(TypedDict):
    analysis_id: str
    status: str
    bbox: str
    fire_date: str


@jobs.tool(annotations=ToolAnnotations(readOnlyHint=False, idempotentHint=False, openWorldHint=True))
def start_burn_analysis(
    bbox: Annotated[str, Field(description="west,south,east,north in WGS-84.")],
    fire_date: Annotated[str, Field(description="Fire date, YYYY-MM-DD.")],
) -> JobTicket:
    """Queue a burn-severity analysis and return its analysis_id at once.

    Pass analysis_id to get_burn_analysis to read the status. An analysis_id
    expires 24 hours after it was created.
    """
    analysis_id = "ana_" + secrets.token_hex(8)
    JOB_STORE[analysis_id] = {"bbox": bbox, "fire_date": fire_date, "status": "queued", "created": time.time()}
    return JobTicket(analysis_id=analysis_id, status="queued", expires_in_hours=24)


@jobs.tool(annotations=ToolAnnotations(readOnlyHint=True, openWorldHint=False))
def get_burn_analysis(analysis_id: Annotated[str, Field(description="The id from start_burn_analysis.")]) -> JobStatus:
    """Return the status and inputs of an analysis started by start_burn_analysis."""
    job = JOB_STORE.get(analysis_id)
    if job is None or time.time() - job["created"] > JOB_TTL_S:
        raise ToolError(f"Unknown or expired analysis_id '{analysis_id}'. Call start_burn_analysis again.")
    return JobStatus(analysis_id=analysis_id, status=job["status"], bbox=job["bbox"], fire_date=job["fire_date"])


async with create_connected_server_and_client_session(jobs) as first_connection:
    started = await first_connection.call_tool(
        "start_burn_analysis", {"bbox": "22.8285,37.9489,24.5933,39.0396", "fire_date": "2021-08-03"}
    )
    handle = started.structuredContent["analysis_id"]
    print("first call:", started.structuredContent)

async with create_connected_server_and_client_session(jobs) as second_connection:
    status = await second_connection.call_tool("get_burn_analysis", {"analysis_id": handle})
    print("second call, new connection:", status.structuredContent)
    stale = await second_connection.call_tool("get_burn_analysis", {"analysis_id": "ana_0000000000000000"})
    print("unknown handle, isError:", stale.isError, "|", stale.content[0].text)

first call: {'analysis_id': 'ana_64102c97ad8035af', 'status': 'queued', 'expires_in_hours': 24}
second call, new connection: {'analysis_id': 'ana_64102c97ad8035af', 'status': 'queued', 'bbox': '22.8285,37.9489,24.5933,39.0396', 'fire_date': '2021-08-03'}
unknown handle, isError: True | Error executing tool get_burn_analysis: Unknown or expired analysis_id 'ana_0000000000000000'. Call start_burn_analysis again.


The handle comes from `secrets`, so it cannot be guessed from another one. A real job would also record who created it and check that on `get_burn_analysis`.


<a id="sec-4-4"></a>

### 4.4 Credentials on each request

Over Streamable HTTP, authentication is normal HTTP: a bearer token, an API key, or a custom header, sent on every request over HTTPS. A server that needs the caller's own account reads it from the request, not from a session.

Over stdio there is no HTTP request. The host starts the process with the credentials in environment variables, and the server reads them from its environment.

When a server needs a user identity, MCP builds on OAuth. See the [authorization specification](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization). Never pass a secret as a tool argument. Arguments reach the model, the logs, and the conversation history.


<a id="sec-4-5"></a>

### 4.5 When the tool needs the user: elicitation

<div class="alert alert-block alert-info" style="background-color: #e8f1f0; color: #173245; border-left: 5px solid #346562; border-radius: 6px; padding: 10px 16px; margin: 12px 0;">

**Use case.** A fire tool defaults `bbox` to all of Europe. A user asks *"any fires this week?"* without a place. The tool could ask which region, instead of returning 50 fires from six countries.

</div>

**Elicitation** is the protocol feature for this. The server asks the host for a value, the host shows the question to the user, and the answer comes back to the tool. The host must support it.

In the Python SDK, call `await ctx.elicit(message, schema)` inside the tool ([elicitation](https://github.com/modelcontextprotocol/python-sdk/blob/v1.30.0/docs/server.md)). For most tools, a required argument with a clear description is enough: the model asks the user before it calls.


<a id="sec-4-6"></a>

### 4.6 Tool annotations: what the call will do

Annotations are hints in `tools/list` that a host may use to decide whether to ask the user first. They are not enforced, and the specification says a client must treat them as untrusted unless the server is trusted.

- `readOnlyHint: true`: the tool does not change anything outside. Example: a geocoding or search tool, `lookup_aoi`.
- `destructiveHint: true`: the tool may delete or overwrite. It is meaningful only when the tool is not read-only. Example: a tool that deletes a stored artifact.
- `idempotentHint: true`: calling twice with the same arguments has no extra effect. Example: `lookup_aoi`.
- `openWorldHint: true`: the tool talks to an outside system whose content changes. Example: a web search, a geocoder.

Set them with `@mcp.tool(annotations=ToolAnnotations(...))`. Section 4.3 does so in code. A read-only tool is the easy case, and it lets a host skip a confirmation dialog.


<a id="sec-4-7"></a>

### 4.7 Extensions to the protocol

Extensions are optional additions that go beyond the core specification. They may be modular, specialised, or experimental. This section follows the [extensions overview](https://modelcontextprotocol.io/extensions/overview).

**How extensions work**

- Each extension has an identifier, `{vendor-prefix}/{extension-name}`. Official ones use `io.modelcontextprotocol`, for example `io.modelcontextprotocol/tasks`. A third-party extension uses a reversed domain name that its author owns.
- An extension is *experimental*, *beta*, or *stable*. The stage tells you how much the design may still change.
- Support is opt-in at every stage. An SDK may not implement a given extension, so check your SDK's documentation.
- Client and server announce the extensions they support in their capabilities, in the `extensions` field.
- When one side does not support an extension, the other falls back to core behaviour or rejects the request. A server should still return useful text for a client that does not support its extension.

**Official extensions worth knowing**

- **MCP Tasks.** Asynchronous execution for long-running operations, with polling and durable handles. It addresses the problem that section 4.3 solves by hand with a start tool and a read tool. Beta.
- **MCP Apps.** A server can display interactive interface elements, such as charts or forms, inline in the conversation. Stable.
- **Skills over MCP.** A server publishes workflow instructions and supporting files as resources. Beta.
- **Server Cards.** A client discovers a server's metadata over HTTP before it connects. Beta.
- **Authorization extensions.** OAuth client credentials for machine-to-machine access, and enterprise-managed authorization. Stable.

Stages change over time. Check the overview page before you depend on an extension.


<a id="sec-4-8"></a>

### 4.8 Tool discoverability

When a host is connected to many servers, it may see dozens of tools. These two patterns are host behaviour, from the [client best practices](https://modelcontextprotocol.io/docs/develop/clients/client-best-practices). Your server still exposes ordinary tools. The host decides when to show them to the model, and whether to chain them in a sandbox.

Loading every schema into every conversation costs tokens before the user has typed anything.

- **Progressive discovery.** The host still calls `tools/list`. It injects a full schema into the model only when the task needs that tool.
- **Programmatic tool calling.** The model writes code that runs several calls in a sandbox. Intermediate results stay there. The model sees only the final summary.

![Loading every tool definition up front, against discovering tools as the task needs them](images/progressive-discovery.svg)

*Fig. 11. Progressive discovery. Source: MCP client best practices.*

![Direct tool calling against a sandbox that runs several calls and returns a short result](images/programmatic-tool-calling.svg)

*Fig. 12. Programmatic tool calling. Source: MCP client best practices.*

Both patterns are a reason to keep each tool's description accurate and each result small. A host can only search for a tool from the description you wrote. It can only type a tool's result in a sandbox if you publish an `outputSchema`.


<a id="sec-terms"></a>

## Terms

- **Agent**: a language model that decides which tools to call, reads what came back, and repeats until it can answer.
- **MCP host**: the application the person is talking to, such as Claude, Cursor, or EVE.
- **MCP client**: the component inside the host that holds one connection to one server. A host that uses three servers has three clients.
- **MCP server**: a program that offers tools and, optionally, resources and prompts.
- **Primitive**: one of the three things a server offers: a tool, a resource, or a prompt.
- **Transport**: how messages move between client and server: stdio or Streamable HTTP.
- **Handle**: an opaque identifier a tool returns so that a later call can refer to earlier work.
- **Registry**: a catalogue of published servers. In this repository, a published server is its folder name under `servers/`.


<a id="sec-resources"></a>

## Resources

**MCP**

- [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture)
- [Understanding MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/learn/server-concepts)
- [MCP specification, 2026-07-28](https://modelcontextprotocol.io/specification/2026-07-28), with its [tools](https://modelcontextprotocol.io/specification/2026-07-28/server/tools), [resources](https://modelcontextprotocol.io/specification/2026-07-28/server/resources), and [prompts](https://modelcontextprotocol.io/specification/2026-07-28/server/prompts) pages
- [MCP extensions overview](https://modelcontextprotocol.io/extensions/overview)
- [Client best practices](https://modelcontextprotocol.io/docs/develop/clients/client-best-practices)

**Building and testing**

- [Building MCP servers](https://github.com/modelcontextprotocol/python-sdk/blob/v1.30.0/docs/server.md) in the Python SDK, v1.30.0: structured output, `ToolError`, context, elicitation
- [MCP Inspector](https://modelcontextprotocol.io/docs/tools/inspector)

**This repository**

- [CONTRIBUTING.md](../CONTRIBUTING.md): the steps to add a server
- [PR #41](https://github.com/eve-esa/mcp-tool-registry/pull/41) and [PR #43](https://github.com/eve-esa/mcp-tool-registry/pull/43): the GDACS server, and its first fix
- [Citation File Format 1.2.0](https://github.com/citation-file-format/citation-file-format/blob/1.2.0/schema-guide.md)
- [Returning files from a tool](https://github.com/eve-esa/backend/blob/main/docs/mcp-file-return-convention.md): how EVE stores an inline image or a resource link


<a id="sec-appendix"></a>

## Appendix



<a id="sec-a"></a>

### A. Existing MCP catalogues

Catalogues and published MCP servers. `EO` marks Earth observation entries. `general` marks the rest.

- [Official MCP Registry](https://registry.modelcontextprotocol.io/) (general)
- [MCP Registry (docs / about)](https://modelcontextprotocol.io/registry/about) (general)
- [MCP Registry (source repo)](https://github.com/modelcontextprotocol/registry) (general)
- [Official MCP reference servers](https://github.com/modelcontextprotocol/servers) (general)
- [Glama MCP Registry](https://glama.ai/mcp) (general)
- [Smithery](https://smithery.ai/) (general)
- [MCP.so](https://mcp.so/) (general)
- [PulseMCP](https://www.pulsemcp.com/) (general)
- [MCP Market](https://mcpmarket.com/) (general)
- [Docker MCP Catalog](https://hub.docker.com/mcp) (general)
- [Docker MCP Registry (source)](https://github.com/docker/mcp-registry) (general)
- [Awesome MCP Servers (punkpeye / mcpservers.org)](https://mcpservers.org/) (general)
- [Awesome MCP Servers (punkpeye GitHub)](https://github.com/punkpeye/awesome-mcp-servers) (general)
- [Awesome MCP Servers (wong2)](https://github.com/wong2/awesome-mcp-servers) (general)
- [Awesome MCP Servers (appcypher)](https://github.com/appcypher/awesome-mcp-servers) (general)
- [Awesome MCP Servers (daedalus)](https://github.com/daedalus/awesome-mcp-servers) (general)
- [Awesome Remote MCP Servers](https://github.com/Appnova-EU-OU/awesome-remote-mcp-servers) (general)
- [freemcp.space](https://freemcp.space/) (general)
- [MCP Find](https://www.mcpfind.org/) (general)
- [MCP Hunt](https://mcphunt.com/) (general)
- [MCP Server Finder](https://www.mcpserverfinder.com/) (general)
- [AIBase MCP Directory](https://mcp.aibase.com/) (general)
- [OpenTools](https://opentools.ai/) (general)
- [LangChain MCP adapters](https://github.com/langchain-ai/langchain-mcp-adapters) (general)
- [ChatForest — Aerospace & Defense MCP Servers](https://chatforest.com/reviews/aerospace-defense-mcp-servers/) (EO)
- [EVE Tool Registry (eve-esa/mcp-tool-registry)](https://github.com/eve-esa/mcp-tool-registry) (EO)
- [NASA Earthdata MCP](https://github.com/nasa/earthdata-mcp) (EO)
- [IBM chuk-mcp-stac](https://github.com/IBM/chuk-mcp-stac) (EO)
- [Copernicus MCP (wb1016)](https://github.com/wb1016/copernicus-mcp) (EO)
- [Wildfire MCP Server](https://github.com/aliafsahnoudeh/wildfire-mcp-server) (EO)
- [GeoSight MCP](https://github.com/armaasinghn/geosight-mcp) (EO)
- [Google Earth Engine MCP (cameronking4)](https://github.com/cameronking4/google-earth-engine-mcp) (EO)
- [GEE MCP (Frontier Development Lab)](https://github.com/FrontierDevelopmentLab/gee-mcp) (EO)
- [Axion MCP (Earth Engine)](https://github.com/Dhenenjay/Axion-MCP) (EO)
- [geeViz (GEE community; includes MCP)](https://github.com/gee-community/geeViz) (EO)
- [STAC MCP (Wayfinder Foundry)](https://github.com/Wayfinder-Foundry/stac-mcp) (EO)
- [openEO + STAC MCP (MatMatt)](https://github.com/MatMatt/openeo-mcp) (EO)
- [EOPF STAC MCP](https://github.com/sebastien-tetaud/eopf-mcp) (EO)
- [GDAL MCP (Wayfinder Foundry)](https://github.com/Wayfinder-Foundry/gdal-mcp) (EO)
- [OpenLandMap MCP](https://github.com/tharlestsa/openlandmap_mcp) (EO)
- [NetCDF MCP (dogukanteber)](https://github.com/dogukanteber/netcdf-mcp) (EO)
- [NetCDF MCP (HanchaoLi/nc-mcp)](https://github.com/HanchaoLi/nc-mcp) (EO)
- [foehn (STAC + xarray/Zarr/NetCDF; includes MCP)](https://github.com/kayhendriksen/foehn) (EO)
- [MapHub (OGC WMS/WFS/WMTS MCP)](https://github.com/YQP-2511/maphub-core) (EO)
- [SkyFi MCP Server](https://github.com/PSkinnerTech/SkyFi-MCP-server) (EO)
- [EEA GeoNetwork MCP](https://github.com/eea/eea-geonetwork-mcp) (EO)
- [Jupyter Earth MCP Server (Datalayer)](https://github.com/datalayer/jupyter-earth-mcp-server) (EO)
- [JAXA Earth MCP](https://github.com/a-hikata/jaxa-earth-mcp) (EO)
- [Groundstation MCP](https://github.com/dannybauman/groundstation) (EO)


<a id="sec-b"></a>

### B. Possible services that could be integrated

Services a future server could wrap. CDSE, EFFIS, and GDACS already have a server in this registry.

- [Copernicus Data Space Ecosystem (CDSE)](https://dataspace.copernicus.eu/) — data, generic. Free Sentinel data. STAC, openEO, OGC, or S3.
- [EFFIS - European Forest Fire Information System](https://forest-fire.emergency.copernicus.eu/) — data, disasters_geohazards. European fire danger, active fires, and burnt area. WMS or a file download.
- [Destination Earth (DestinE)](https://destination-earth.eu/) — data + model, atmosphere_climate. EU climate digital twins and a data lake. Harmonised Data Access REST API.
- [Copernicus Land Monitoring Service (CLMS)](https://land.copernicus.eu/) — data, land. Land cover, vegetation, water, and soil moisture. OGC services or a file download.
- [Copernicus Marine Service (CMEMS)](https://marine.copernicus.eu/) — data, marine_maritime. Ocean forecasts and reanalysis. Copernicus Marine Toolbox.
- [Copernicus Emergency Management Service (CEMS)](https://emergency.copernicus.eu/) — data, disasters_geohazards. On-demand maps for floods, fires, and earthquakes. Activations REST API.
- [CEMS Early Warning Data Store - EFAS / GloFAS](https://ewds.climate.copernicus.eu/) — data, disasters_geohazards. River-discharge forecasts for Europe and the globe. cdsapi client.
- [Copernicus Climate & Atmosphere Data Store (CDS / ADS)](https://cds.climate.copernicus.eu/) — data, atmosphere_climate. ERA5 climate and atmospheric composition. cdsapi client.
- [ESA Application Propagation Environments (APEx)](https://apex.esa.int/) — platform (algorithm + data catalogue), generic. Research algorithms offered as on-demand services. openEO or OGC API Processes.
- [EO Training Data Lab (EOTDL)](https://eotdl.com/) — data + model, generic. Training datasets and pretrained EO models. REST API or the Python library.
- [CREODIAS](https://creodias.eu/) — data + compute, generic. Copernicus archive next to cloud compute. S3, STAC, or openEO.
- [WEkEO](https://www.wekeo.eu/) — data + compute, generic. Sentinel, marine, and climate data behind one API, plus cloud processing. HDA client.
- [ESA WorldCereal](https://esa-worldcereal.org/) — data + model, land. Global crop-type maps and reference data. REST API or openEO.
- [Google Earth Engine (GEE)](https://earthengine.google.com/) — data + model (compute platform), generic. Public satellite catalog and server-side analysis. Python or JavaScript client.
- [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/) — data + compute, generic. Open catalog of Sentinel, Landsat, and climate data. STAC API.
- [Sentinel Hub (Sinergise, a Planet company)](https://www.sentinel-hub.com/) — data (processing service), generic. On-the-fly imagery and analytics. OGC services or the Process API.
- [Earth Search (Element 84)](https://earth-search.aws.element84.com/) — data (catalogue), generic. STAC search over Sentinel and Landsat on AWS.
- [Maxar](https://www.maxar.com/) — data (commercial), generic. Commercial very-high-resolution imagery and tasking. OGC services or the Python SDK.
- [Airbus Intelligence (OneAtlas)](https://www.intelligence.airbus.com/) — data (commercial), generic. Commercial Pléiades and SPOT imagery. Data REST API.
- [Capella Space](https://www.capellaspace.com/) — data (commercial), disasters_geohazards. Commercial SAR tasking and archive. REST API.
- [ICEYE](https://www.iceye.com/) — data (commercial), disasters_geohazards. Commercial SAR, plus flood extent and depth. Catalog and tasking REST APIs.
- [Clay Foundation Model](https://clay-foundation.github.io/model/) — model + embedding, generic. Open model that embeds satellite imagery. Weights on Hugging Face.
- [NASA Earthdata](https://www.earthdata.nasa.gov/) — data, generic. NASA satellite archive. CMR search or CMR-STAC.
- [NSIDC - National Snow and Ice Data Center](https://nsidc.org/) — data, snow_ice. Snow, ice, and permafrost. earthaccess library.
- [USGS EarthExplorer](https://earthexplorer.usgs.gov/) — data, land. Landsat and other USGS datasets. M2M API.
- [EUMETSAT Data Store](https://data.eumetsat.int/) — data, atmosphere_climate. Meteosat, Metop, and Sentinel ocean and atmosphere products. EUMDAC client.
- [JAXA G-Portal / JAXA Earth API](https://www.eorc.jaxa.jp/) — data, land; disasters_geohazards. ALOS, GCOM, and GSMaP. Earth API or G-Portal.
- [Radiant Earth / Source Cooperative (successor to Radiant MLHub)](https://source.coop/) — data, generic. Open geospatial and ML training datasets. S3.
- [FAO SEPAL](https://sepal.io/) — platform (no-code compute + data), land. Cloud notebooks for forest monitoring, on top of Earth Engine. Web UI.
- [Planet Labs](https://www.planet.com/) — data (commercial), generic. Daily optical imagery and sub-meter tasking. Data and Orders APIs.
- [Global Forest Watch (World Resources Institute)](https://www.globalforestwatch.org/) — data (derived/thematic), land. Tree-cover change and deforestation alerts. REST API.
- [GBIF - Global Biodiversity Information Facility](https://www.gbif.org/) — data, biodiversity_ecosystems. Species occurrence records. REST API.
- [OBIS - Ocean Biodiversity Information System](https://obis.org/) — data, biodiversity_ecosystems. Marine species occurrences. REST API.
- [Protected Planet / WDPA (UNEP-WCMC)](https://www.protectedplanet.net/) — data, biodiversity_ecosystems. Global protected areas. REST API or a shapefile download.
- [Global Mangrove Watch](https://www.globalmangrovewatch.org/) — data, biodiversity_ecosystems. Global mangrove extent and change. WMS or a GeoTIFF download.
- [EMODnet - European Marine Observation and Data Network](https://emodnet.ec.europa.eu/) — data, marine_maritime. European bathymetry, habitats, and marine human activity. OGC services.
- [Global Fishing Watch](https://globalfishingwatch.org/) — data, marine_maritime. Vessel tracks and fishing activity. REST API.
- [GDACS - Global Disaster Alert and Coordination System](https://www.gdacs.org/) — data (derived/thematic), disasters_geohazards. Earthquake, cyclone, flood, and volcano alerts. GeoJSON API, no account.
- [Overture Maps Foundation](https://overturemaps.org/) — data, built_environment. Open buildings, roads, and places. GeoParquet on S3.
- [Google Open Buildings](https://sites.research.google/gr/open-buildings/) — data, built_environment. Building footprints for Africa, Latin America, and Asia. Earth Engine or a CSV download.
- [Microsoft Global ML Building Footprints](https://github.com/microsoft/GlobalMLBuildingFootprints) — data, built_environment. Global building footprints. GeoJSON download, or Planetary Computer.
